<a href="https://colab.research.google.com/github/satyamroy3913-maker/CurrencyConverters/blob/main/Day02_Goal02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic pandas

In [ ]:
import os
from getpass import getpass

key = None
for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):   # dono naam try karta hai
    try:
        from google.colab import userdata
        key = userdata.get(name)
        print(f"Secret '{name}' mil gaya ✅")
        break
    except Exception:
        continue

if not key:
    key = getpass("Gemini API key paste karo: ")

os.environ["GEMINI_API_KEY"] = key

# folder structure
os.makedirs("/content/chaining/prompts", exist_ok=True)
os.makedirs("/content/chaining/briefs", exist_ok=True)
os.makedirs("/content/chaining/runs", exist_ok=True)

%cd /content/chaining
print("Ready ✅")

Cell 3: Step 1 prompt, prompts/extract.txt

In [ ]:
%%writefile /content/chaining/prompts/extract.txt
You are a data extraction assistant for an advertising team.
Read the product brief below and extract exactly these fields:
product_name, category, key_features, target_audience.

Rules:
- Use ONLY information stated in the brief. Never invent anything.
- If a text field is not stated in the brief, write "NOT_SPECIFIED".
- If no features are stated, return an empty list for key_features.
- key_features: short plain-fact phrases (max 8 words each), 2 to 6 items.
- Do not copy marketing superlatives (best, cheapest, #1) into features. State the plain fact instead.

PRODUCT BRIEF:
"""
{brief}
"""

Cell 4: Step 2 prompt, prompts/generate.txt

In [ ]:
%%writefile /content/chaining/prompts/generate.txt
You are a copywriter for an advertising agency.
Write 3 DIFFERENT ad copy variants for the product described in the JSON below.

Use a different angle for each variant:
1. benefit-led (what the customer gains)
2. feature-led (what the product does)
3. lifestyle/emotion-led (how it fits the customer's life)

Rules:
- Use ONLY facts present in the JSON. Do not invent features, prices, numbers or awards.
- headline: max 10 words. body: max 30 words, ending with a short call to action.
- Friendly, clear tone. Write for the target_audience.
- If target_audience is "NOT_SPECIFIED", write for a general audience. Do not guess a specific group.
- Give each variant an id (1, 2, 3) and name its angle.

PRODUCT ATTRIBUTES (JSON):
{attributes_json}

Cell 5: Step 3 prompt, prompts/select.txt

In [ ]:
%%writefile /content/chaining/prompts/select.txt
You are a brand compliance reviewer.
Pick the single best ad variant from the 3 below, using ONLY the brand rules and the fact check.

How to decide:
1. Reject any variant that breaks a brand rule (banned words, invented facts, length limits).
2. Reject any variant that claims something not present in the product attributes.
3. From the remaining variants, pick the one that is clearest and most useful for the target audience.
4. If every variant breaks a rule, pick the least bad one and say clearly which rule it breaks.

Return winner_id, a short explanation (why), and for every variant you did NOT pick, a one-line reason.

BRAND RULES:
{brand_rules}

PRODUCT ATTRIBUTES (JSON), for fact checking:
{attributes_json}

AD VARIANTS (JSON):
{variants_json}

Cell 6: brand_rules.md

In [ ]:
%%writefile /content/chaining/brand_rules.md
# Brand Rules (used in Step 3)

## 1. Banned words (no superlatives or absolute claims)
These words must never appear in an ad. The pipeline also checks them in plain Python after Step 3.
- best
- cheapest
- #1
- number one
- unbeatable
- ultimate
- perfect
- guaranteed
- finest
- world-class
- revolutionary

## 2. No invented facts
Every claim must come from the product attributes. No made-up numbers, prices, awards or comparisons with competitors.

## 3. Length
- Headline: maximum 10 words
- Body: maximum 30 words

## 4. Call to action
The body must end with a simple call to action (e.g. "Try it today", "Order now").

## 5. Tone
Friendly, clear, honest. No fear tactics, no pressure ("last chance!!!").

Cell 7: 5 product briefs (fictional brands)

In [ ]:
briefs = {
    1: """We are launching AquaPure Mini, a pocket-sized water purifier bottle for hikers and college students who travel a lot. It filters out bacteria and sediment using a replaceable carbon filter, weighs only 250 grams, and charges by USB-C. One charge lasts about a week of daily use. Priced around Rs 2,499. Tone should be adventurous but friendly.""",

    2: """Product: VoltRide E1 electric scooter. Made for office-goers in metro cities who are tired of traffic and fuel prices. Top speed 45 km/h, range of 80 km per charge, removable battery you can charge at your desk, and a keyless unlock through the phone app. Comes with a 3 year battery warranty. Launch campaign for Diwali season.""",

    3: """ChaiLeaf Organics is a monthly tea subscription box. Every month we send 4 small-batch teas from Assam and Darjeeling gardens, with a short card about the farmer who grew them. Packaging is fully compostable. Customers can pause or cancel anytime from the app. Our buyers are mostly working professionals aged 25-40 who love tea but are bored of supermarket bags.""",

    4: """PocketLedger - an expense tracking app for freelancers and small creators. It reads bank SMS alerts and auto-sorts spending into business and personal, makes GST-ready invoices in 2 taps, and sends a reminder when a client payment is overdue. There is a free plan and a paid plan at Rs 199 a month. Earlier we also sold a budgeting notebook, but ignore that, this campaign is only about the app.""",

    5: """Need ad copy for our new thing, Glowkind. We think it is India's best skincare, cheapest in the market, and guaranteed to work for everyone. Not sure about the exact ingredients yet, marketing team will confirm later. Launching soon, maybe festive season? Make it sound premium but also affordable.""",
}

for n, text in briefs.items():
    with open(f"/content/chaining/briefs/brief_{n}.txt", "w", encoding="utf-8") as f:
        f.write(text)

print("5 briefs saved ✅")

In [ ]:

# pipeline.py ka default bhi update kar do
p = "/content/chaining/pipeline.py"
s = open(p).read()
s = s.replace('"gemini-2.5-flash"', f'"{NEW_MODEL}"')
open(p, "w").write(s)

# already imported module ko reload karo
sys.path.insert(0, "/content/chaining")
import pipeline
importlib.reload(pipeline)

Cell 8: pipeline.py

In [ ]:
%%writefile /content/chaining/pipeline.py
"""
Prompt chaining pipeline: Extract -> Generate -> Select

    python pipeline.py                      # all 5 briefs
    python pipeline.py --only 3             # only brief_3
    python pipeline.py --mock               # offline test, no API calls
    python pipeline.py --only 2 --corrupt-step1 --no-guard --runs-dir runs_experiment
"""
import argparse
import json
import os
import re
import time
from pathlib import Path

from pydantic import BaseModel

BASE = Path(__file__).parent
PROMPTS_DIR = BASE / "prompts"
BRIEFS_DIR = BASE / "briefs"
MODEL = os.environ.get("MODEL", "gemini-2.5-flash")


# ---------------- Schemas (response_schema) ----------------
class ProductAttributes(BaseModel):
    product_name: str
    category: str
    key_features: list[str]
    target_audience: str


class AdVariant(BaseModel):
    id: int
    angle: str
    headline: str
    body: str


class Variants(BaseModel):
    variants: list[AdVariant]


class Rejected(BaseModel):
    id: int
    reason: str


class Winner(BaseModel):
    winner_id: int
    why: str
    rejected: list[Rejected]


# ---------------- Helpers ----------------
def load_prompt(name: str) -> str:
    return (PROMPTS_DIR / f"{name}.txt").read_text(encoding="utf-8")


def fill(template: str, **values) -> str:
    # str.replace, because JSON has { } braces and str.format would break
    for key, value in values.items():
        template = template.replace("{" + key + "}", value)
    return template


def save_json(path: Path, data) -> None:
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def load_banned_words() -> list[str]:
    text = (BASE / "brand_rules.md").read_text(encoding="utf-8")
    section = text.split("## 1.")[1].split("## 2.")[0]
    return [line[2:].strip().lower() for line in section.splitlines() if line.startswith("- ")]


# ---------------- LLM call ----------------
_client = None


def _get_client():
    global _client
    if _client is None:
        from google import genai

        key = os.environ.get("GEMINI_API_KEY")
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            raise RuntimeError("GEMINI_API_KEY not found.")
        _client = genai.Client(api_key=key)
    return _client


def call_llm(step, prompt, schema, temperature, mock=False, mock_ctx=None) -> dict:
    if mock:
        return mock_llm(step, mock_ctx or {})

    from google.genai import types

    last_err = None
    for attempt in range(3):
        try:
            resp = _get_client().models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=temperature,
                ),
            )
            return json.loads(resp.text)
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM call failed after 3 attempts: {last_err}")


def mock_llm(step, ctx) -> dict:
    """Fake outputs, only to test the plumbing offline."""
    if step == "step1":
        return {"product_name": ctx["brief"].split()[0], "category": "mock",
                "key_features": ["mock feature one", "mock feature two"], "target_audience": "mock"}
    if step == "step2":
        n = ctx["attributes"]["product_name"]
        return {"variants": [
            {"id": 1, "angle": "benefit", "headline": f"Meet {n}", "body": "Made for you. Try it today."},
            {"id": 2, "angle": "feature", "headline": f"{n} does more", "body": "Simple and useful. Order now."},
            {"id": 3, "angle": "lifestyle", "headline": f"{n} fits your day", "body": "Easy to love. Buy now."}]}
    return {"winner_id": 1, "why": "Mock decision.", "rejected": [{"id": 2, "reason": "mock"}, {"id": 3, "reason": "mock"}]}


# ---------------- Validation between steps ----------------
def words(text: str) -> set:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 3}


def validate_attributes(attrs: dict, brief: str):
    """Returns (errors, warnings). Errors stop the chain, warnings are only recorded."""
    errors, warnings = [], []
    name = attrs.get("product_name", "").strip()
    if not name or name == "NOT_SPECIFIED":
        errors.append("product_name missing")
    elif name.lower() not in brief.lower():
        errors.append(f"product_name '{name}' does not appear in the brief (possible hallucination)")

    cat = attrs.get("category", "").strip()
    if not cat or cat == "NOT_SPECIFIED":
        errors.append("category missing")

    feats = attrs.get("key_features", [])
    if len(feats) < 2:
        errors.append(f"only {len(feats)} key feature(s) extracted, need at least 2")
    else:
        bw = words(brief)
        ungrounded = [f for f in feats if not (words(f) & bw)]
        if len(ungrounded) > len(feats) / 2:
            warnings.append(f"most features share no words with the brief: {ungrounded}")

    if attrs.get("target_audience", "").strip() in ("", "NOT_SPECIFIED"):
        warnings.append("target_audience not specified in brief")
    return errors, warnings


def validate_variants(data: dict):
    errors = []
    vs = data.get("variants", [])
    if len(vs) != 3:
        errors.append(f"expected 3 variants, got {len(vs)}")
    for v in vs:
        if len(v["headline"].split()) > 10:
            errors.append(f"variant {v['id']}: headline longer than 10 words")
        if len(v["body"].split()) > 30:
            errors.append(f"variant {v['id']}: body longer than 30 words")
    return errors


def check_banned_words(variants, banned):
    """Plain-Python check, we do not trust the LLM alone to catch superlatives."""
    found = {}
    for v in variants:
        text = f"{v['headline']} {v['body']}".lower()
        found[str(v["id"])] = [b for b in banned if re.search(r"(?<!\w)" + re.escape(b) + r"(?!\w)", text)]
    return found


# ---------------- One brief through the chain ----------------
WRONG_ATTRS = {
    "category": "credit card",
    "key_features": ["5% cashback on flights", "free airport lounge access"],
    "target_audience": "frequent international flyers",
}


def run_one(brief_path: Path, run_dir: Path, args, banned) -> dict:
    run_dir.mkdir(parents=True, exist_ok=True)
    brief = brief_path.read_text(encoding="utf-8").strip()
    (run_dir / "input_brief.txt").write_text(brief, encoding="utf-8")
    brand_rules = (BASE / "brand_rules.md").read_text(encoding="utf-8")

    status = {"brief": brief_path.name, "model": "mock" if args.mock else MODEL, "steps": {}, "halted_at": None}

    def finish():
        save_json(run_dir / "status.json", status)
        return status

    # STEP 1: Extract
    try:
        t0 = time.time()
        p1 = fill(load_prompt("extract"), brief=brief)
        attrs = call_llm("step1", p1, ProductAttributes, 0.0, args.mock, {"brief": brief})
    except Exception as e:
        status["steps"]["step1"] = {"ok": False, "error": str(e)}
        status["halted_at"] = "step1"
        return finish()

    if args.corrupt_step1:
        attrs = {**attrs, **WRONG_ATTRS}
        status["step1_corrupted_on_purpose"] = True

    save_json(run_dir / "step1_attributes.json", attrs)
    errs, warns = validate_attributes(attrs, brief)
    status["steps"]["step1"] = {"ok": not errs, "errors": errs, "warnings": warns, "seconds": round(time.time() - t0, 2)}
    if errs and not args.no_guard:
        status["halted_at"] = "step1"
        return finish()

    # STEP 2: Generate
    try:
        t0 = time.time()
        p2 = fill(load_prompt("generate"), attributes_json=json.dumps(attrs, indent=2, ensure_ascii=False))
        variants = call_llm("step2", p2, Variants, 0.9, args.mock, {"attributes": attrs})
    except Exception as e:
        status["steps"]["step2"] = {"ok": False, "error": str(e)}
        status["halted_at"] = "step2"
        return finish()

    save_json(run_dir / "step2_variants.json", variants)
    errs2 = validate_variants(variants)
    status["steps"]["step2"] = {"ok": not errs2, "errors": errs2, "seconds": round(time.time() - t0, 2)}
    if errs2 and not args.no_guard:
        status["halted_at"] = "step2"
        return finish()

    # STEP 3: Select
    try:
        t0 = time.time()
        p3 = fill(load_prompt("select"), brand_rules=brand_rules,
                  attributes_json=json.dumps(attrs, indent=2, ensure_ascii=False),
                  variants_json=json.dumps(variants, indent=2, ensure_ascii=False))
        decision = call_llm("step3", p3, Winner, 0.0, args.mock, {})
    except Exception as e:
        status["steps"]["step3"] = {"ok": False, "error": str(e)}
        status["halted_at"] = "step3"
        return finish()

    violations = check_banned_words(variants["variants"], banned)
    winner_hits = violations.get(str(decision["winner_id"]), ["(winner id not found)"])
    save_json(run_dir / "step3_winner.json", {
        "llm_decision": decision,
        "code_check": {
            "banned_words_found_per_variant": violations,
            "winner_has_banned_words": bool(winner_hits),
            "winner_banned_words": winner_hits,
        },
    })
    status["steps"]["step3"] = {
        "ok": not winner_hits,
        "errors": [f"winner contains banned words: {winner_hits}"] if winner_hits else [],
        "seconds": round(time.time() - t0, 2),
    }
    return finish()


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--only", type=int)
    ap.add_argument("--mock", action="store_true")
    ap.add_argument("--no-guard", action="store_true")
    ap.add_argument("--corrupt-step1", action="store_true")
    ap.add_argument("--runs-dir", default="runs")
    args = ap.parse_args(argv)

    banned = load_banned_words()
    runs_root = BASE / args.runs_dir
    briefs = sorted(BRIEFS_DIR.glob("brief_*.txt"))
    if args.only:
        briefs = [b for b in briefs if b.stem == f"brief_{args.only}"]

    summary = []
    for b in briefs:
        n = b.stem.split("_")[1]
        print(f"\n=== {b.name} ===")
        st = run_one(b, runs_root / f"run_{n}", args, banned)
        for step, info in st["steps"].items():
            flag = "OK  " if info.get("ok") else "FAIL"
            print(f"  {step}: {flag} {info.get('errors') or info.get('error') or ''}"
                  f" {('warn: ' + str(info['warnings'])) if info.get('warnings') else ''}")
        if st["halted_at"]:
            print(f"  -> chain halted at {st['halted_at']}")
        summary.append(st)

    save_json(runs_root / "summary.json", summary)
    print(f"\nDone. Outputs in {runs_root}/")


if __name__ == "__main__":
    main()

In [ ]:
# pipeline.py ka default bhi update kar do
p = "/content/chaining/pipeline.py"
s = open(p).read()
s = s.replace('"gemini-2.5-flash"', f'"{NEW_MODEL}"')
open(p, "w").write(s)

# already imported module ko reload karo
sys.path.insert(0, "/content/chaining")
import pipeline
importlib.reload(pipeline)


Cell 9: Pehle sirf Step 1 test karo

In [ ]:
import os, re, sys, importlib, subprocess, time
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# 1) aisa model dhundo jiska quota bacha ho
EXHAUSTED = "gemini-3.8-flash"      # jiska quota khatam hai
candidates = []
for m in client.models.list():
    name = m.name.replace("models/", "")
    actions = getattr(m, "supported_actions", None) or []
    if ("flash" in name and "generateContent" in actions and name != EXHAUSTED
            and not any(x in name for x in ("image", "tts", "live", "audio"))):
        candidates.append(name)
print("Try karenge:", candidates[:8])

working = None
for name in candidates[:8]:
    try:
        client.models.generate_content(model=name, contents="Say OK")
        print(f"✅ {name} chal raha hai")
        working = name
        break
    except Exception as e:
        print(f"❌ {name}: {str(e)[:120]}")

assert working, "Kisi model pe quota nahi bacha. 1 ghanta ruko ya nayi key (naya project) banao."

# 2) pipeline.py mein model set karo, aur quota error pe retry band karo
os.environ["MODEL"] = working
p = "/content/chaining/pipeline.py"
s = open(p).read()
s = re.sub(r'os\.environ\.get\("MODEL",\s*"[^"]*"\)', f'os.environ.get("MODEL", "{working}")', s)
old = "        except Exception as e:\n            last_err = e\n            time.sleep(2 * (attempt + 1))"
new = ("        except Exception as e:\n            last_err = e\n"
       "            if 'RESOURCE_EXHAUSTED' in str(e) or 'NOT_FOUND' in str(e):\n                break\n"
       "            time.sleep(2 * (attempt + 1))")
if old in s:
    s = s.replace(old, new)
open(p, "w").write(s)
print("Model set:", working)

# 3) jo briefs baaki hain wo chalao
for n in range(1, 6):
    if os.path.exists(f"/content/chaining/runs/run_{n}/step3_winner.json"):
        print(f"brief {n}: pehle se ho chuka, skip")
        continue
    r = subprocess.run(["python", "/content/chaining/pipeline.py", "--only", str(n)],
                       capture_output=True, text=True, cwd="/content/chaining")
    print(r.stdout[-600:] or r.stderr[-600:])
    if "RESOURCE_EXHAUSTED" in r.stdout + r.stderr:
        print("⚠️ Quota khatam. Baad mein ye cell dobara chalao, bacha hua kaam wahin se hoga.")
        break
    time.sleep(15)

Cell 12: Results table

In [ ]:
import json, os
import pandas as pd

rows = []
for n in range(1, 6):
    d = f"/content/chaining/runs/run_{n}"
    if not os.path.exists(f"{d}/status.json"):
        rows.append({"brief": n, "halted_at": "NOT RUN YET"})
        continue
    status = json.load(open(f"{d}/status.json"))
    row = {"brief": n, "halted_at": status["halted_at"]}
    for s in ("step1", "step2", "step3"):
        info = status["steps"].get(s)
        row[s] = None if info is None else ("OK" if info["ok"] else "FAIL")
    if os.path.exists(f"{d}/step3_winner.json"):
        w = json.load(open(f"{d}/step3_winner.json"))
        row["winner_id"] = w["llm_decision"]["winner_id"]
        row["winner_banned_words"] = w["code_check"]["winner_banned_words"]
    row["step1_warnings"] = status["steps"].get("step1", {}).get("warnings")
    rows.append(row)

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

Cell 13: Ek run ko detail mein dekho

In [ ]:
import os

def show_run(n):
    d = f"/content/chaining/runs/run_{n}"
    for title, fname in [("INPUT", "input_brief.txt"), ("STEP 1", "step1_attributes.json"),
                         ("STEP 2", "step2_variants.json"), ("STEP 3", "step3_winner.json"),
                         ("STATUS", "status.json")]:
        path = f"{d}/{fname}"
        if os.path.exists(path):
            print(f"{title}:\n{open(path).read()}\n")
        else:
            print(f"{title}: (file nahi bani, chain pehle ruk gayi)\n")

show_run(5)

In [ ]:
import os, shutil, subprocess

d = "/content/chaining/runs/run_5"
if not os.path.exists(f"{d}/step3_winner.json"):
    if os.path.exists(d):
        shutil.rmtree("/content/chaining/brief5_halted_copy", ignore_errors=True)
        shutil.copytree(d, "/content/chaining/brief5_halted_copy")
        print("Ruka hua version bacha liya: brief5_halted_copy/")
    r = subprocess.run(["python", "/content/chaining/pipeline.py", "--only", "5", "--no-guard"],
                       capture_output=True, text=True, cwd="/content/chaining")
    print(r.stdout[-800:] or r.stderr[-800:])
else:
    print("run_5 pehle se poora hai, kuch karne ki zarurat nahi ✅")

Cell 14: Error propagation experiment

In [ ]:
!python /content/chaining/pipeline.py --only 2 --corrupt-step1 --no-guard --runs-dir runs_experiment

Cell 15: Experiment ka output compare karo

In [ ]:
import json, os

d = "/content/chaining/runs_experiment/run_2"
if not os.path.exists(f"{d}/step3_winner.json"):
    print("Experiment poora nahi hua. Cell 14 ka output dekho:")
    print(open(f"{d}/status.json").read() if os.path.exists(f"{d}/status.json") else "run_2 folder hi nahi bana")
else:
    print("Galat Step 1 attributes:\n", open(f"{d}/step1_attributes.json").read())

    variants = json.load(open(f"{d}/step2_variants.json"))
    print("\nStep 2 ne kya likha:")
    for v in variants["variants"]:
        print(f"[{v['id']}] {v['headline']} | {v['body']}")

    w = json.load(open(f"{d}/step3_winner.json"))
    print("\nStep 3 ka decision:\n", json.dumps(w["llm_decision"], indent=2, ensure_ascii=False))

    print("\nStep 1 status (warnings dekho):")
    print(json.dumps(json.load(open(f"{d}/status.json"))["steps"]["step1"], indent=2))

Cell 16: learnings.md

In [ ]:
%%writefile /content/chaining/learnings.md
# Learnings: Prompt Chaining (Day 2 GOAL2)

## 1. Results overview
| Brief | Step 1 | Step 2 | Step 3 | Winner id | Banned words in winner? |
|-------|--------|--------|--------|-----------|--------------------------|
| 1 AquaPure Mini | | | | | |
| 2 VoltRide E1 | | | | | |
| 3 ChaiLeaf Organics | | | | | |
| 4 PocketLedger | | | | | |
| 5 Glowkind (messy) | | | | | |

## 2. What broke during testing
- [Kaunsa brief/step pehle fail hua? Exact error ya galat output kya tha?]
- [Brief 4 mein "ignore the notebook" line se Step 1 confuse hua? Brief 5 mein model ne NOT_SPECIFIED likha ya kuch invent kiya?]

## 3. Error-propagation experiment (--corrupt-step1)
- [Kya Step 2 phir bhi "work" kiya? Scooter ke ads mein kya likha tha?]
- [Kya Step 3 ne galat facts notice kiye? Phir bhi winner pick kiya?]
- [Kya status.json ke warning ne galti pakdi? Kaunsa check?]

## 4. Error handling across steps
- [Silent failure vs loud failure: crash dikh jaata hai, galat-par-valid JSON nahi.]
- [Steps ke beech validation kahan kaam aaya: schema, grounding check, banned-word check?]
- [Banned words ke liye LLM checker vs plain Python checker: kaun zyada reliable raha?]
- [Har intermediate file save karne se debugging kaise aasaan hui.]

## 5. Splitting tasks into smaller prompts
- [Har prompt ka kaam ek line mein. Kaunsa fix karna sabse aasaan tha?]
- [Ek bada prompt kaisa rehta? (Optional: brief 5 pe try karke compare karo.)]
- [Cost/latency trade-off: 1 call ki jagah 3 calls.]

Cell 17: Sab kuch zip karke download karo

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/chaining_submission", "zip", "/content", "chaining")
files.download("/content/chaining_submission.zip")